# Lección 05 - RAG Agente


## Configuración

Este cuaderno demuestra el patrón Agentic RAG (Generación Aumentada por Recuperación) usando el Microsoft Agent Framework.

**Requisitos previos:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — el endpoint de tu servicio Azure AI Search
- `AZURE_SEARCH_API_KEY` — tu clave API de Azure AI Search
- Despliegue de Azure OpenAI configurado mediante variables de entorno
- Azure CLI autenticado (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ¿Qué es Agentic RAG?

El RAG tradicional sigue una secuencia fija: recuperar documentos y luego generar una respuesta. **Agentic RAG** va más allá al darle al agente la autonomía para decidir **cuándo** y **cómo** recuperar información.

Con Agentic RAG, el agente puede:
- **Decidir** si es necesario recuperar información antes de responder una pregunta
- **Elegir** qué fuente de datos o herramienta consultar
- **Evaluar** los resultados recuperados y realizar recuperaciones adicionales si el primer intento no es suficiente
- **Combinar** información de múltiples pasos de recuperación en una respuesta coherente

Esto hace que el agente sea más flexible y preciso en comparación con una secuencia estática de recuperar y luego generar.


## Creación de una herramienta de búsqueda

En Agentic RAG, las fuentes de datos externas se envuelven como **herramientas** que el agente puede invocar bajo demanda. Esto permite que el agente trate la recuperación como otra acción más que puede realizar, en lugar de un paso obligatorio.

A continuación, definimos una base de conocimientos de viajes y la exponemos como una herramienta que el agente puede usar para consultar información sobre destinos.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Construyendo el Agente RAG

Ahora creamos un agente que está instruido para **siempre recuperar información antes de responder**. El agente utiliza la herramienta `search_travel_knowledge` para fundamentar sus respuestas en la base de conocimientos en lugar de confiar en sus propios datos de entrenamiento.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Recuperación Iterativa — El Patrón Maker-Checker

Una ventaja clave de Agentic RAG es la **recuperación iterativa**. El agente puede realizar múltiples rondas de búsqueda para verificar, refinar o ampliar sus hallazgos iniciales — similar a un flujo de trabajo "maker-checker":

1. **Paso Maker**: El agente recupera información inicial y redacta una respuesta.
2. **Paso Checker**: El agente realiza recuperaciones adicionales para verificar detalles o llenar vacíos.

A continuación, se le plantea al agente una pregunta que requiere comparar múltiples destinos, lo que lo impulsa a buscar varias veces.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Resumen

En esta lección aprendiste cómo construir un sistema **Agentic RAG** utilizando el Microsoft Agent Framework:

- **Agentic RAG** permite que los agentes decidan de forma autónoma cuándo recuperar información, haciendo que la recuperación sea dinámica en lugar de fija.
- **Herramientas como fuentes de datos**: Las bases de conocimiento externas (como Azure AI Search) se envuelven como herramientas que el agente puede invocar.
- **Recuperación iterativa**: El patrón maker-checker permite que el agente realice múltiples rondas de recuperación — buscando, verificando y refinando — antes de producir una respuesta final.

En producción, reemplazarías la base de conocimiento en memoria `TRAVEL_KNOWLEDGE_BASE` con un índice real de Azure AI Search para manejar la recuperación de documentos de viaje a gran escala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por humanos. No nos hacemos responsables de malentendidos o interpretaciones erróneas que puedan derivarse del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
